# 1 · Análise exploratória

**Tech Challenge Fase 3 — Predição e inteligência analítica para alfabetização**

Este notebook percorre a análise exploratória que sustenta as decisões de modelagem.
As funções vêm de `src/visualization/eda.py`; aqui está a narrativa.

Pergunta de fundo: *onde está a variação da alfabetização, e qual parte dela é
previsível a partir de dados públicos?*

In [ ]:
import sys, warnings
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))
warnings.filterwarnings("ignore", category=FutureWarning)

# O acesso ao BigQuery exige o token de curta duração:
#   export GCP_ACCESS_TOKEN=$(gcloud auth print-access-token)

from src.modeling.dataset import carregar_abt
from src.preprocessing.features import ALVO, blocos, selecionar_features
from src.visualization import eda

df = carregar_abt()
features = selecionar_features(list(df.columns))
df.shape, len(features)

## A base

`features.abt_aluno` tem um registro por aluno **efetivamente avaliado** — quem faltou
à prova foi excluído, porque para esses o rótulo é determinístico (ver
`docs/decisoes-analiticas.md` §1) e o modelo aprenderia apenas a detectar ausência.

In [ ]:
for bloco, cols in sorted(blocos(features).items(), key=lambda kv: -len(kv[1])):
    print(f"{bloco:32} {len(cols):3d} features")

df.groupby("ano")[ALVO].agg(alunos="size", taxa="mean")

## Distribuições: por ano, rede e região

In [ ]:
eda.fig_distribuicao_alvo(df)

## Onde está a dispersão

A variação entre escolas é maior que entre municípios — mas, como a próxima seção
mostra, quase toda ela é transitória.

In [ ]:
eda.fig_dispersao_unidades(df)

## O que persiste de um ano para o outro

Este é o achado que determina o desenho do projeto: **a taxa municipal é altamente
persistente (r ≈ 0,81)**.

O valor calculado para "escola" (r ≈ 0,22) **não** mede persistência escolar:
`id_escola` é renumerado a cada ano — só 2,4% dos identificadores presentes em 2023 e
2024 apontam para o mesmo município. Unir os anos por essa chave liga escolas
diferentes. Por isso nenhuma feature longitudinal de escola entra no projeto.

In [ ]:
caminho, persistencia = eda.fig_persistencia(df)
persistencia

## Correlações

Medidas no grão do **município**. No grão do aluno, uma variável municipal é constante
dentro do município e sua correlação é diluída pelo ruído individual. A leitura
municipal é uma correlação **ecológica**: descreve diferenças entre redes de ensino,
e não pode ser lida como efeito sobre um indivíduo.

In [ ]:
mun = eda.agregar_municipio(df)
corr = eda.correlacoes_municipais(mun, features)
corr.head(20)

In [ ]:
eda.fig_top_correlacoes(corr)

## Formato da relação

Decis das variáveis mais associadas ao alvo. Curvas monotônicas indicam relação
estável; curvas em U revelam não linearidade, que só um modelo flexível captura.

In [ ]:
principais = [c for c in corr.head(6)["feature"] if c in df.columns][:4]
eda.fig_decis(df, principais)

## Geografia

In [ ]:
eda.fig_mapa(mun)

## Dados ausentes

A ausência se concentra nas features defasadas e **não é aleatória**: marca municípios
que entraram na avaliação em 2024. Por isso a imputação vem acompanhada de um
indicador binário de ausência — "sem histórico" é informação, não ruído.

In [ ]:
eda.fig_ausencias(df, features)

## Hipóteses que a exploração sustenta

1. **O componente previsível é municipal.** A persistência entre anos existe no
   município (r = 0,81) e não há como medi-la na escola.
2. **Contexto socioeconômico e demográfico pesa mais que infraestrutura escolar.**
   As correlações mais fortes são demográficas e de vulnerabilidade, não de
   equipamento.
3. **A predição individual terá teto baixo e a agregada será boa.** A variação
   dentro de um mesmo município é grande e não observável com dados públicos; a
   média municipal, ao contrário, é bem determinada.
4. **O teste de 2024 mistura duas capacidades**: generalizar no tempo e extrapolar
   para UFs novas (AC, DF, SP). As métricas precisam ser estratificadas.